INSTALL LIBRARIES

In [1]:
!pip install transformers datasets pandas torch rouge_score sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 37.5 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ad6aaf967213c6573db87f29185ffa460a3b0b90846c3e6c89f0b4646988a4c9
  

In [2]:
!pip install accelerate

In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


IMPORT LIBRARIES

In [4]:
import os
import pandas as pd
import numpy as np
import string
import re
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    MvpTokenizer,
    MvpForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    GenerationConfig,#########
    EarlyStoppingCallback,
)
from evaluate import load as load_metric

In [5]:
# import os
# import pandas as pd
# import numpy as np
# import string
# import re
# import torch
# from datasets import Dataset, DatasetDict, load_metric
# from transformers import (
#     MvpTokenizer,
#     MvpForConditionalGeneration,
#     DataCollatorForSeq2Seq,
#     Seq2SeqTrainingArguments,
#     Seq2SeqTrainer,
#     GenerationConfig,#########
#     EarlyStoppingCallback,
# )

REMOVE MARKERS FUNCTION

*   Removes markers
*   Strip whitespace



In [6]:

def clean_tagged_dataframe(
    df: pd.DataFrame,
    cols_to_clean: list = ["source", "target"],
    tags: list = None
) -> pd.DataFrame:
    """
    Remove predefined tags from specified text columns in a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        cols_to_clean (list of str): Column names to clean (default ["source","target"]).
        tags (list of regex strings): Tags to remove. If None, defaults to:
            [r'\[INTRO\]', r'\[QUES\]', r'\[VITALS\]', r'\[HIST\]', r'\[SYM\]'].

    Returns:
        pd.DataFrame: A copy of the DataFrame with the specified columns cleaned.
    """
    # Default tags if none provided
    if tags is None:
        tags = [r'\[INTRO\]',r'\[INRO\]', r'\[QUES\]', r'\[VITALS\]', r'\[HIST\]', r'\[SYM\]']
    pattern = "|".join(tags)

    df_clean = df.copy()
    cleaned_any = False

    for col in cols_to_clean:
        if col in df_clean.columns:
            # ensure text, strip tags & whitespace
            df_clean[col] = (
                df_clean[col]
                .astype(str)
                .str.replace(pattern, "", regex=True)
                .str.strip()
            )
            cleaned_any = True

    if not cleaned_any:
        raise ValueError(
            f"None of the columns {cols_to_clean} found in DataFrame. "
            f"Found columns: {list(df_clean.columns)}"
        )

    return df_clean

CLEAN TEXT FUNCTION


*   Removes unwanted items
*   Removes non-alphanumeric characters



In [7]:

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Replace '-' with a space
    text = text.replace("-", " ")
    text = text.replace("/", " ")
    text = text.replace("+", " ")
    text = text.replace("?", " ")
    text = text.replace("!", " ")
    text = text.replace(",", " ")
    text = text.replace(":", " ")
    text = text.replace(";", " ")
    text = text.replace("?"," ")
    text = text.replace("%", " ")
    text = text.replace("*"," ")
    text = text.replace("°C"," ")
    text = text.replace("."," ")
    # Remove punctuation
    translator = str.maketrans("", "", string.punctuation)
    text = text.translate(translator)
    # Remove any other non-alphanumeric characters
    text = re.sub(r"[^a-z0-9\s]", "", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

LOAD DATA


*   Load datasets
*   Preview dataframes



In [9]:
# 1. Load data
train_df = pd.read_csv("/content/extracted_marker_train_json_data_clean.csv").dropna(subset=["source","target"]).reset_index(drop=True)
test_df  = pd.read_csv("/content/extracted_marker_test_json_data_clean.csv").dropna(subset=["source"]).reset_index(drop=True)

In [10]:
train_df = clean_tagged_dataframe(train_df)
test_df = clean_tagged_dataframe(test_df)

In [11]:
train_df

,source,target
0,A 4-year-old child presents to the emergency d...,Summary:\nA 4 year old with 5% superficial bur...
1,A 6 year old girl presented to the emergency d...,Summary\n6-year-old present with vomiting and ...
2,"Forty-seven years, old man, came to Casualty, ...",Summary\nA 47-year-old man presents with sever...
3,"ER, aged 72 years, female was brought in with...",SUMMARY\n\n72-year-old female with inability t...
4,A 22-year-old female patient is brought in wi...,"A 22 year old female presents with headache, d..."
...,...,...
400,A 48 years old lady was admitted with complain...,Summary\n48-year-old female admitted with easy...
401,A 25-year-old man presents to the OPD with com...,SUMMARY\n25 yr old man with known rheumatic he...
402,"A 69 yr old male presents at opd with fever, ...","SUMMARY\n69-year-old male with fever, chest pa..."
403,A 3 year old female is brought with complaints...,SUMMARY\n3-year-old female with increased thir...


In [12]:
test_df

,source
0,A 24 year old female complains of sharp pain i...
1,"A 3 years old boy was brought to the facility,..."
2,"A 22-year-old man, was brought in by the moth..."
3,A 6 years old girl is brought to Opd with his...
4,"So, I was in the MCH clinic in the morning. I ..."
...,...
95,A child 6 years of age came to the clinic acco...
96,A 49-year-old lady presented at the facility w...
97,A 35 year old man comes to the outpatient depa...
98,A 48 year old was brought from theatre post th...


In [13]:
test_df['source'] = test_df['source'].str.replace(r"[-\n]", " ", regex=True)
train_df['source'] = train_df['source'].str.replace(r"[-\n]", " ", regex=True)
train_df['target'] = train_df['target'].str.replace(r"[-\n]", " ", regex=True)


In [14]:
train_df

,source,target
0,A 4 year old child presents to the emergency d...,Summary: A 4 year old with 5% superficial burn...
1,A 6 year old girl presented to the emergency d...,Summary 6 year old present with vomiting and a...
2,"Forty seven years, old man, came to Casualty, ...",Summary A 47 year old man presents with severe...
3,"ER, aged 72 years, female was brought in with...",SUMMARY 72 year old female with inability to ...
4,A 22 year old female patient is brought in wi...,"A 22 year old female presents with headache, d..."
...,...,...
400,A 48 years old lady was admitted with complain...,Summary 48 year old female admitted with easy ...
401,A 25 year old man presents to the OPD with com...,SUMMARY 25 yr old man with known rheumatic hea...
402,"A 69 yr old male presents at opd with fever, ...","SUMMARY 69 year old male with fever, chest pai..."
403,A 3 year old female is brought with complaints...,SUMMARY 3 year old female with increased thirs...


In [15]:
test_df

,source
0,A 24 year old female complains of sharp pain i...
1,"A 3 years old boy was brought to the facility,..."
2,"A 22 year old man, was brought in by the moth..."
3,A 6 years old girl is brought to Opd with his...
4,"So, I was in the MCH clinic in the morning. I ..."
...,...
95,A child 6 years of age came to the clinic acco...
96,A 49 year old lady presented at the facility w...
97,A 35 year old man comes to the outpatient depa...
98,A 48 year old was brought from theatre post th...


TRAIN TEST SPLIT

*   Split Test data
*   Split Train data



In [16]:
from sklearn.model_selection import train_test_split

# Split train_df into train and test parts
train, test = train_test_split(
    train_df,
    test_size=0.2,        # 20% goes to test_df
    random_state=42,      # for reproducibility
    shuffle=True          # shuffle before splitting (default)
)

CONVERT DATAFRAMES INTO HUGGINGFACE DATASET FORMAT

In [17]:
hf_train = Dataset.from_pandas(train)
hf_test  = Dataset.from_pandas(test)

LOAD MODEL AND TOKENIZER

In [18]:
# 2. Tokenizer & Model
tokenizer = MvpTokenizer.from_pretrained("RUCAIBox/mvp")
model     = MvpForConditionalGeneration.from_pretrained("RUCAIBox/mtl-data-to-text")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

CHECK MAXIMUM TOKEN COUNT IN SOURCE AND TAGET OF TRAIN/TEST

In [20]:
# from transformers import AutoTokenizer

# # Load the tokenizer you're using
# tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

# Count tokens per example
train_df['source_token_count'] = train_df['source'].apply(lambda x: len(tokenizer.encode(x, truncation=False)))
train_df['target_token_count'] = train_df['target'].apply(lambda x: len(tokenizer.encode(x, truncation=False)))

# Get maximums
max_source_length = train_df['source_token_count'].max()
max_target_length = train_df['target_token_count'].max()

print(f"Max source length: {max_source_length} tokens")
print(f"Max target length: {max_target_length} tokens")


Max source length: 291 tokens
Max target length: 500 tokens


In [21]:
# Count tokens per example
test_df['source_token_count'] = test_df['source'].apply(lambda x: len(tokenizer.encode(x, truncation=False)))

# Get maximums
max_source_length_t = test_df['source_token_count'].max()

print(f"Max source length: {max_source_length_t} tokens")

Max source length: 275 tokens


In [18]:
# max_input_length  = 512 #512
# max_output_length = 256  #256 #128 # 256

SET MAXIMUM TOKEN: INPUT AND OUTPUT LENGTH

In [22]:
max_input_length  = 384    # >291, so no source will be truncated
max_output_length = 512    # covers your 500-token targets


PREPROCESSING FUNCION

In [23]:
# 3. Preprocessing fn
def preprocess_fn(batch):
    inputs = tokenizer(
        batch["source"],
        truncation=True,
        max_length=max_input_length,
        padding="max_length",
    )
    targets = tokenizer(
        batch["target"],
        truncation=True,
        max_length=max_output_length,
        padding="max_length",
    ) if "target" in batch else {}
    labels = targets.get("input_ids", None)
    # replace pad token id's in labels by -100 so they are ignored by loss
    if labels is not None:
        labels = [
            [(l if l != tokenizer.pad_token_id else -100) for l in lab]
            for lab in labels
        ]
    return {
        **inputs,
        **({"labels": labels} if labels is not None else {}),
    }

In [20]:
# 4. Prepare tokenized datasets
# tokenized_datasets = DatasetDict({
#     "train": hf_train.map(preprocess_fn, batched=True, remove_columns=hf_train.column_names),
#     "eval":  hf_train.shuffle().select(range(100)).map(preprocess_fn, batched=True, remove_columns=hf_train.column_names),
# })

APPLY PREPROCESSING FUNCTION

In [24]:
# from datasets import DatasetDict

tokenized_datasets = DatasetDict({
    "train": hf_train.map(
        preprocess_fn,
        batched=True,
        remove_columns=hf_train.column_names
    ),
    "eval": hf_test.map(
        preprocess_fn,
        batched=True,
        remove_columns=hf_test.column_names
    ),
})


Map:   0%|          | 0/324 [00:00<?, ? examples/s]

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

DATA COLLATOR

In [25]:
# 5. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

ROUGE METRIC

In [26]:
# 6. Load ROUGE
rouge = load_metric("rouge")


ROUGE METRIC FUNCTION

In [27]:


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}


CUDA CLEAR CACHE


*   Had a few memory issues
*   Did not have good resources



In [28]:
import torch
torch.cuda.empty_cache()


In [27]:
# training_args = Seq2SeqTrainingArguments(
#     # ← your generation settings bundled here
#     generation_config=GenerationConfig(
#         decoder_start_token_id=tokenizer.pad_token_id,
#         max_length=512,           # allow up to 512 tokens in the answer
#         min_length=256,           # ensure at least 128 tokens are produced
#         length_penalty=1.2,       # discourage too-short outputs
#         num_beams=4,              # use beam search for more coherent long output
#         no_repeat_ngram_size=3,   # avoid repeating the same 3-gram
#         early_stopping=True,      # stop once all beams finish
#     ),

#     output_dir="mvp-models",

#     # → epoch-based evaluation & checkpointing
#     eval_strategy="epoch",    # run eval at the end of each epoch
#     save_strategy="epoch",          # save a checkpoint at the end of each epoch
#     logging_strategy="epoch",
#     load_best_model_at_end=True,    # after training, reload the best checkpoint on your metric
#     metric_for_best_model="eval_rouge1",  # or whichever ROUGE key you prefer
#     greater_is_better=True,              # for ROUGE f‑scores

#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=4,  # or more
#     learning_rate=5e-5,
#     weight_decay=0.01,
#     num_train_epochs=20,

#     predict_with_generate=True,         # enable generation during eval
#     #generation_max_length=512,          # passed to model.generate()
#     #generation_num_beams=4,             # passed to model.generate()
#     #fp16=True,                          # if you have a GPU
#     push_to_hub=False,
#     report_to="none",
#     #torch_compile=False,           # disable torch.compile calls inside Trainer
# )

SEQ2SEQ TRAIN ARGS SETUP

In [29]:
training_args = Seq2SeqTrainingArguments(

    generation_config=GenerationConfig(
        decoder_start_token_id=tokenizer.pad_token_id,
        max_length=512,           # allow up to 512 tokens in the answer
        min_length=256,           # *256 ensure at least 256 tokens are produced
        length_penalty=1.2,       # discourage too-short outputs
        num_beams=4,              # use beam search for more coherent long output
        no_repeat_ngram_size=3,   # avoid repeating the same 3-gram
        early_stopping=True,      # stop once all beams finish
    ),

    # → epoch-based evaluation & checkpointing
    eval_strategy="steps",    # run eval at the end of each epoch
    save_strategy="steps",          # save a checkpoint at the end of each epoch
    logging_strategy="steps",
    output_dir="model-mvp",
    load_best_model_at_end=True,    # after training, reload the best checkpoint on your metric
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    metric_for_best_model="eval_rouge1",  # or whichever ROUGE key you prefer
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=20,

    predict_with_generate=True,         # enable generation during eval
    #generation_max_length=512,          # passed to model.generate()
    #generation_num_beams=4,             # passed to model.generate()
    #fp16=True,                          # if you have a GPU
    push_to_hub=False,
    report_to="none",

)

In [30]:
# 8. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset= tokenized_datasets["eval"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,     # use ROUGE
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5, early_stopping_threshold=0.001)
    ]
)

/tmp/ipython-input-30-1533709742.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


ENABLE MODEL GRADIENT CHECKPOINTING

In [31]:
model.gradient_checkpointing_enable()  # at model instantiation

TRAIN MODEL

In [32]:
# 9. Train + evaluate
trainer.train()
# during evaluation steps, trainer will:
#   * call model.generate(...) with your beam/length settings
#   * compute ROUGE on the generated outputs

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
500,0.938700,3.373022,0.275300,0.106100,0.177900,0.177700,511.000000


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3685: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 5, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=820, training_loss=1.2859045970730665, metrics={'train_runtime': 3557.622, 'train_samples_per_second': 1.821, 'train_steps_per_second': 0.23, 'total_flos': 5266064183132160.0, 'train_loss': 1.2859045970730665, 'epoch': 20.0})

METRICS

In [ ]:
# 10. Final evaluation on held-out eval split
metrics = trainer.evaluate()
print("Eval metrics:", metrics)

APPLY PREDICTIONS ON TEST DATAFRAME

In [ ]:
# 11. Inference on test set
model.eval()
all_preds = []
batch_size = 8
for i in range(0, len(test_df), batch_size):
    batch_src = test_df["source"].iloc[i : i + batch_size].tolist()
    inputs = tokenizer(
        batch_src,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
        padding="longest",
    ).to(trainer.args.device)

    generated_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=256,           # match your target length
        min_length=80,            # avoid too‑short outputs
        length_penalty=1.2,       # encourage full summarization
        num_beams=10,              # more thorough search
        no_repeat_ngram_size=3,   # reduce loops
        early_stopping=True,
        bad_words_ids=[[tokenizer.encode(w)[0]] for w in ["basically", "actually", "..."]]
    )
    preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    all_preds.extend(preds)

TRAIN TESTING

In [53]:
import numpy as np
from evaluate import load as load_metric

# Only take first 5 examples
subset_df = train_df.iloc[:4]

model.eval()
all_preds_train = []
batch_size = 8

for i in range(0, len(subset_df), batch_size):
    batch_src = subset_df["source"].iloc[i : i + batch_size].tolist()
    inputs = tokenizer(
        batch_src,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
        padding="longest",
    ).to(trainer.args.device)

    # generated_ids = model.generate(
    #     inputs["input_ids"],
    #     attention_mask=inputs["attention_mask"],
    #     max_length=512,
    #     min_length=256,
    #     length_penalty=1.2,
    #     num_beams=4,
    #     no_repeat_ngram_size=3,
    #     early_stopping=True,
    # )

    generated_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=256,           # match your target length
        min_length=80,            # avoid too‑short outputs
        length_penalty=1.2,       # encourage full summarization
        num_beams=10,              # more thorough search
        no_repeat_ngram_size=3,   # reduce loops
        early_stopping=True,
        bad_words_ids=[[tokenizer.encode(w)[0]] for w in ["basically", "actually", "..."]]
    )

    preds_train = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    all_preds_train.extend(preds_train)

# Now all_preds_train contains your 5 generated summaries
for idx, (src, pred) in enumerate(zip(subset_df["source"], all_preds_train), 1):
    print(f"Example {idx}\nSource: {src}\nGenerated: {pred}\n{'-'*60}")



Example 1
Source: A 4 year old child presents to the emergency department with second degree burns on the forearm after accidentally touching a hot stove. The child was playing in the kitchen when they reached out to touch the stove.  The burns cover about 5% of the total body surface area. The child is alert and crying, with redness, blisters, and swelling on the affected area. The burns appear to be superficial to moderate in severity. The child is in mild pain, and there is no indication of airway or breathing distress.  No other injuries are noted.  Questions: 1. What is the immediate treatment protocol for second degree burns in paediatric patients? 2. Should any tetanus prophylaxis be considered in this case? 3. What follow up care should be recommended for burn healing?
Generated: Summary A 4 year old child presents with second degree burns on the forearm after accidentally touching a hot stove. The burns cover about 5% of the body surface area and are superficial to moderate in

In [41]:
# 2. Prepare references
references = train_df["target"].tolist()

# 3. Compute ROUGE
rouge = load_metric("rouge")
rouge_res = rouge.compute(
    predictions=all_preds_train,
    references=references,
    use_stemmer=True
)
# extract F‑scores ×100
rouge_scores = {f"rouge_{k}": v * 100 for k, v in rouge_res.items()}

# 4. Exact‑Match Accuracy
exact_matches = [p.strip() == r.strip() for p, r in zip(all_preds_train, references)]
exact_match = 100.0 * sum(exact_matches) / len(exact_matches)

# 5. Token‑Level Accuracy
correct_tokens = 0
total_tokens   = 0
for p, r in zip(all_preds_train, references):
    p_toks = tokenizer.tokenize(p)
    r_toks = tokenizer.tokenize(r)
    L = min(len(p_toks), len(r_toks))
    correct_tokens += sum(pt == rt for pt, rt in zip(p_toks, r_toks))
    total_tokens   += max(len(r_toks), 1)

token_accuracy = 100.0 * correct_tokens / total_tokens

# 6. Ratio of Exact‑Match to Token Accuracy
ratio_em_to_ta = exact_match / token_accuracy

# 7. Print everything
print(f"ROUGE scores: {rouge_scores}")
print(f"Exact‑Match Accuracy: {exact_match:.2f}%")
print(f"Token‑Level Accuracy: {token_accuracy:.2f}%")
print(f"EM / Token‑Acc ratio: {ratio_em_to_ta:.4f}")

ROUGE scores: {'rouge_rouge1': np.float64(33.61123235701714), 'rouge_rouge2': np.float64(20.110621841434956), 'rouge_rougeL': np.float64(25.240889232033403), 'rouge_rougeLsum': np.float64(25.22692247931846)}
Exact‑Match Accuracy: 0.00%
Token‑Level Accuracy: 10.04%
EM / Token‑Acc ratio: 0.0000


In [46]:
train_df['Target_Compare']  = all_preds_train

In [47]:
train_df[['target','Target_Compare']]

,target,Target_Compare
0,Summary: A 4 year old with 5% superficial burn...,Summary A 4 year old child presents with secon...
1,Summary 6 year old present with vomiting and a...,Summary 6 year old presents with vomiting and ...
2,Summary A 47 year old man presents with severe...,Summary A 47 year old man presents with severe...
3,SUMMARY 72 year old female with inability to ...,SUMMARY 72 year old female with inability to w...
4,"A 22 year old female presents with headache, d...",Summary A 22 year old female presents with hea...
...,...,...
400,Summary 48 year old female admitted with easy ...,"SUMMARY 48 year old with easy fatigability, le..."
401,SUMMARY 25 yr old man with known rheumatic hea...,Summary: 25 year old man with a history of run...
402,"SUMMARY 69 year old male with fever, chest pai...","SUMMARY 69 yr old male with fever, chest pains..."
403,SUMMARY 3 year old female with increased thirs...,Summary 3 year old female with complaints of i...


GENERATE SUBMISSION FILE

In [36]:
test_raw_df = pd.read_csv('/content/test_raw.csv')

master_index = test_raw_df[['Master_Index']]

master_index

,Master_Index
0,ID_CUAOY
1,ID_OGSAY
2,ID_TYHSA
3,ID_CZXLD
4,ID_ZJQUQ
...,...
95,ID_AYCAI
96,ID_CLEYN
97,ID_BRIIW
98,ID_SNZBL


In [50]:
# 12. Export submission
submission = test_df.copy()
submission["Clinician"] = all_preds
submission["Master_Index"] = master_index['Master_Index']

submission = submission[["Master_Index","Clinician"]]
submission.to_csv("test_submission_tests.csv", index=False)
print("Saved submission.csv with", len(submission), "rows.")

Saved submission.csv with 100 rows.


In [35]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [38]:
# !cp -r /content/model-mvp /content/drive/MyDrive/

In [ ]:
# import shutil

# # Define source and destination paths
# src = "/content/model-mvp"
# dst = "/content/drive/MyDrive/model-mvp"

# # Copy entire directory tree
# shutil.copytree(src, dst)
